In [1]:
import pandas as pd
import sqlalchemy as sa
from tkinter import ON
from sqlalchemy.sql.operators import op

engine = sa.create_engine("sqlite:////Users/yugaljagtap/Downloads/SQL/olist.db")

def run(query):
    return pd.read_sql(query, engine)

## 1. Order Status Overview

**Business Question:** What order statuses exist in the system and how many orders have each status?

In [ ]:
df = run("""
SELECT order_status, COUNT(order_status) AS num_orders
FROM olist_orders_dataset
GROUP BY order_status
ORDER BY num_orders DESC
""")

df.to_csv('order_status.csv', index=False)
df

,order_status,num_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2



**Findings:**
96% of orders on the Olist platform are successfully delivered, indicating a strong fulfillment operation. However, ~625 cancelled and 609 unavailable orders suggest potential stock availability or logistics issues worth investigating further.

In [8]:
run("SELECT * FROM olist_customers_dataset LIMIT 5")

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


## 2. Customer Distribution by State

**Business Question:** Which Brazilian states have the most customers, and where should the marketing team focus their ad campaign?

In [2]:
df =run("""
SELECT COUNT(customer_id) AS num_customers, customer_state
FROM olist_customers_dataset
GROUP by customer_state
ORDER BY COUNT(customer_id) DESC
""")
df.to_csv('customer_state.csv', index=False)
df

,num_customers,customer_state
0,41746,SP
1,12852,RJ
2,11635,MG
3,5466,RS
4,5045,PR
5,3637,SC
6,3380,BA
7,2140,DF
8,2033,ES
9,2020,GO


**Finding:**
São Paulo (SP) dominates the customer base with 41,746 customers — 
approximately 3x larger than the second biggest state, Rio de Janeiro (RJ) 
with 12,852. The top 5 states (SP, RJ, MG, RS, PR) represent the majority 
of Olist's customer base.

**Recommendation:** Marketing budget should be concentrated in these 5 states 
to maximize reach among existing high-density customer regions, while also 
exploring growth potential in mid-tier states like SC and BA which show 
emerging demand.

## 3. Order Volume by Year

**Business Question:** How has order volume grown year over year since Olist launched?

In [12]:
df =run("""
SELECT  strftime('%Y', order_purchase_timestamp) AS Year, COUNT(order_purchase_timestamp) AS total_order_placed
FROM olist_orders_dataset
GROUP BY  strftime('%Y', order_purchase_timestamp)
""")
df.to_csv('order_volume_by_year.csv', index=False)
df

,Year,total_order_placed
0,2016,329
1,2017,45101
2,2018,54011


**Findings:**
Olist experienced explosive growth between 2016 and 2017, with total orders 
increasing nearly 15x (329 → 45,101). Growth continued into 2018 with 54,011 
orders, though at a steadier pace.

**Note:** 2016 and 2018 data may be incomplete (partial years), which could 
explain the lower order counts for those years. The 2017 figure of 45,101 
is the most reliable full-year benchmark.

## 4. Revenue by Payment Type

**Business Question:** Which payment methods do customers use most, and how much revenue does each generate?

In [13]:
df = run("""
SELECT payment_type, SUM(payment_value) AS total_value
FROM  olist_orders_dataset AS od
JOIN olist_order_payments_dataset AS op ON od.order_id = op.order_id
GROUP BY payment_type
ORDER BY total_value DESC
""")
df.to_csv('payment_type.csv', index=False)
df

,payment_type,total_value
0,credit_card,12542084.19
1,boleto,2869361.27
2,voucher,379436.87
3,debit_card,217989.79
4,not_defined,0.00


**Findings:**
Credit cards dominate Olist's payment landscape, generating 12,542,084 BRL 
in total value — over 10x more than the second largest method, Boleto 
(2,869,361 BRL). Debit cards represent the lowest performing payment type 
at 217,989 BRL, while 'not_defined' entries suggest some data quality issues 
worth investigating.

**Recommendation:** Prioritize a seamless credit card checkout experience 
as it drives the vast majority of revenue. Additionally, investigate the 
'not_defined' payment entries to ensure no revenue is being misclassified.

## 5. Top 10 Product Categories by Revenue

**Business Question:** Which product categories generate the most revenue, 
and where should the product team invest in expanding inventory?

In [15]:
df = run("""
SELECT trans.product_category_name_english, SUM(op.payment_value) AS total_value
FROM olist_order_items_dataset AS oi 
JOIN olist_products_dataset AS  p ON oi.product_id = p.product_id  
JOIN olist_order_payments_dataset AS op ON oi.order_id = op.order_id 
JOIN product_category_name_translation AS trans ON p.product_category_name = trans.product_category_name 
GROUP BY trans.product_category_name_english
ORDER BY total_value DESC LIMIT 10;
""")
df.to_csv('product_category_by_value.csv', index=False)
df

,product_category_name_english,total_value
0,bed_bath_table,1712553.67
1,health_beauty,1657373.12
2,computers_accessories,1585330.45
3,furniture_decor,1430176.39
4,watches_gifts,1429216.68
5,sports_leisure,1392127.56
6,housewares,1094758.13
7,auto,852294.33
8,garden_tools,838280.75
9,cool_stuff,779698.00


**Findings:**
The top 5 revenue-generating categories on Olist are bed_bath_table 
(1,712,553 BRL), health_beauty (1,657,373 BRL), computers_accessories 
(1,585,330 BRL), furniture_decor (1,430,176 BRL) and watches_gifts 
(1,429,216 BRL).

**Recommendation:** Inventory investment should be prioritized in 
bed_bath_table, computers_accessories and furniture_decor as these are 
high-revenue categories with no shelf life concerns. 

**Caution:** Despite being the 2nd highest revenue category, health_beauty 
products carry a significantly shorter shelf life. Overstocking risks 
product expiry and write-offs — a just-in-time inventory approach is 
recommended for this category.

## 6. Underperforming Product Categories by Review Score

**Business Question:** Which product categories have the lowest average customer review scores, and what does this mean for the platform's product quality strategy?

In [16]:
df = run("""
SELECT  trans.product_category_name_english, AVG(orr.review_score) AS average_review , COUNT(orr.review_id) AS total_reviews
FROM olist_order_items_dataset AS oi
JOIN olist_order_reviews_dataset AS orr ON oi.order_id = orr.order_id
JOIN olist_products_dataset AS p ON oi.product_id = p.product_id
JOIN product_category_name_translation AS trans ON p.product_category_name = trans.product_category_name

GROUP BY trans.product_category_name_english
HAVING average_review < 4 AND total_reviews > 50
ORDER BY average_review DESC;
""")
df.to_csv('product_category_by_review.csv', index=False)
df

,product_category_name_english,average_review,total_reviews
0,fashion_underwear_beach,3.976923,130
1,air_conditioning,3.969178,292
2,kitchen_dining_laundry_garden_furniture,3.964286,280
3,telephony,3.946867,4517
4,home_construction,3.940000,600
5,art,3.937198,207
6,computers_accessories,3.930819,7849
7,furniture_living_room,3.904382,502
8,furniture_decor,3.903493,8331
9,bed_bath_table,3.895663,11137


**Findings:** No product category averaged below 3.0 stars — a positive signal for 
Olist's overall product quality. However, filtering for categories with 
more than 10 reviews and average score below 4.0 revealed some nuanced findings.

Notably, 3 of our top 5 revenue-generating categories appear in the below 
4.0 rating group: computers_accessories (3.93, 7,849 reviews), 
furniture_decor (3.90, 8,331 reviews) and bed_bath_table (3.89, 11,137 reviews). 
Despite the lower scores, their high review counts indicate strong sales volume, these categories are not failing, but there is room for improvement.

**Recommendation:** A deeper investigation is needed into whether the 
dissatisfaction in these top categories is logistics-related (late delivery, 
damaged goods) or product-related (quality, wrong description). Resolving 
this could significantly boost revenue further.

**Warning:** fashion_male_clothing (3.64, 131 reviews) and 
fashion_underwear_beach (3.97, 130 reviews) show both low ratings and 
low review counts, indicating weak customer engagement and poor satisfaction. 
These categories should be monitored closely for potential removal from 
the platform.

## 7. Late Deliveries by Customer State
**Business Question:** The logistics team wants to identify orders where the actual delivery date was later than the estimated delivery date, to measure how often Olist fails to meet its delivery promises.


In [17]:
df = run("""
SELECT cd.customer_state, COUNT(cd.customer_state) AS total_state_count_late_deliveries
FROM olist_orders_dataset AS od
JOIN olist_customers_dataset AS cd ON od.customer_id = cd.customer_id
WHERE order_status = 'delivered' AND order_delivered_customer_date > order_estimated_delivery_date
GROUP BY cd.customer_state
ORDER BY total_state_count_late_deliveries DESC
LIMIT 20;
""")
df.to_csv('late_deliveries.csv', index=False)
df

,customer_state,total_state_count_late_deliveries
0,SP,2387
1,RJ,1664
2,MG,637
3,BA,457
4,RS,382
5,SC,346
6,PR,246
7,ES,244
8,CE,196
9,PE,172


**Findings:** The top 3 states for late deliveries — São Paulo (2,387), Rio de Janeiro 
(1,664) and Minas Gerais (637) — mirror the top 3 states by customer 
volume, meaning their high late delivery counts are partially explained 
by their larger customer base.

However, Bahia (BA) and Santa Catarina (SC) stand out as disproportionately 
affected. With customer bases of 3,380 and 3,637 respectively, their late 
delivery counts of 457 and 346 represent a significantly higher late 
delivery ratio compared to other states of similar size.

**Recommendation:** While logistics improvements are needed nationwide, 
BA and SC should be prioritized for immediate investigation as they show 
signs of a structural logistics problem beyond what customer volume alone 
can explain.

### 7.1 Late Deliveries Rate %

In [6]:
run("""
SELECT 
    cd.customer_state, 
    COUNT(CASE WHEN od.order_delivered_customer_date > od.order_estimated_delivery_date THEN 1 END) AS late_orders,
    COUNT(od.order_id) AS total_orders,
    ROUND(CAST(COUNT(CASE WHEN od.order_delivered_customer_date > od.order_estimated_delivery_date THEN 1 END) AS FLOAT) / COUNT(od.order_id) * 100, 2) AS late_ratio_percentage
FROM olist_orders_dataset AS od
JOIN olist_customers_dataset AS cd ON od.customer_id = cd.customer_id
WHERE od.order_status = 'delivered'
GROUP BY cd.customer_state
ORDER BY late_ratio_percentage DESC
LIMIT 20;
""")

,customer_state,late_orders,total_orders,late_ratio_percentage
0,AL,95,397,23.93
1,MA,141,717,19.67
2,PI,76,476,15.97
3,CE,196,1279,15.32
4,SE,51,335,15.22
5,BA,457,3256,14.04
6,RJ,1664,12350,13.47
7,TO,35,274,12.77
8,PA,117,946,12.37
9,ES,244,1995,12.23


**Findings:** While São Paulo (SP) and Rio de Janeiro (RJ) have the highest total counts of late deliveries, this is largely a function of their massive order volume. When calculating the Late Delivery Ratio, we discover that Northern and Northeastern states like Alagoas (AL) and Maranhão (MA) often experience significantly higher percentage-based failure rates.

**Recommendations:** For high-risk regions, Olist should consider adjusting "Estimated Delivery Dates" to be more conservative, reducing the gap between promise and reality to protect customer satisfaction scores.

## 8. Order Distribution by Spending Tier
**Business Question:** The marketing team wants to segment customers based on their spending behavior to create targeted campaigns.

In [18]:
df = run("""
SELECT payment_tier, COUNT(payment_tier) AS total_orders
FROM(SELECT payment_value,
CASE
	WHEN payment_value < 100 THEN "Low"
	WHEN payment_value BETWEEN 100 AND 500 THEN "Medium"
	ELSE "High"
END AS payment_tier
FROM olist_order_payments_dataset
)
	GROUP BY payment_tier
	ORDER BY total_orders DESC
""")
df.to_csv('payment_tier.csv', index=False)
df

,payment_tier,total_orders
0,Low,51855
1,Medium,47775
2,High,4256


**Findings:** Olist's order distribution skews heavily toward low and medium value 
purchases — Low tier (below 100 BRL) accounts for 51,855 orders, Medium 
(100-500 BRL) for 47,775, while High value orders (above 500 BRL) 
represent only 4,256 orders.

This gap likely reflects typical e-commerce behavior where customers 
prefer purchasing high value items in physical stores due to trust and 
inspection concerns.

**Recommendation:** To close the gap between Medium and High tier orders, 
Olist should consider:
1. **Review analysis** — Investigate complaints from existing high value 
   customers to identify barriers to purchase
2. **Competitive pricing** — Compare high value item prices against offline 
   market rates and offer incentivized pricing
3. **Logistics improvement** — Ensure premium handling and packaging for 
   high value items to build customer trust in online purchasing

## 9. Top 10 Sellers by Revenue with State Ranking
**Business Question:** The sales team wants to understand each seller's performance by ranking them based on total revenue generated.

In [19]:
df = run("""
SELECT id.seller_id, SUM(pd.payment_value ) AS total_payment_value, DENSE_RANK() OVER (ORDER BY  SUM(pd.payment_value ) DESC) AS ranking, sd.seller_state
FROM olist_order_items_dataset AS id
JOIN olist_order_payments_dataset AS pd ON id.order_id = pd.order_id
JOIN olist_sellers_dataset AS sd ON id.seller_id = sd.seller_id
GROUP BY id.seller_id
ORDER BY total_payment_value DESC 
LIMIT 20;
""")
df.to_csv('seller_ranking_by_payment_value_state.csv', index=False)
df

,seller_id,total_payment_value,ranking,seller_state
0,7c67e1448b00f6e969d365cea6b010ab,507166.91,1,SP
1,1025f0e2d44d7041d6cf58b6550e0bfa,308222.04,2,SP
2,4a3ca9315b744ce9f8e9374361493884,301245.27,3,SP
3,1f50f920176fa81dab994f9023523100,290253.42,4,SP
4,53243585a1d6dc2643021fd1853d8905,284903.08,5,BA
5,da8622b14eb17ae2831f4ac5b9dab84a,272219.32,6,SP
6,4869f7a5dfa277a7dca6462dcf3b52b2,264166.12,7,SP
7,955fee9216a65b617aa5c0531780ce60,236322.30,8,SP
8,fa1c13f2614d7b5c4749cbc52fecda94,206513.23,9,SP
9,7e93a43ef30c4f03f38b393420bc753a,185134.21,10,SP


**Findings:** The top 10 sellers by revenue are heavily dominated by São Paulo (SP) 
based sellers, with 8 out of 10 top earners operating from SP. The #1 
seller generated 507,166 BRL — nearly 65% more than the #2 seller 
(308,222 BRL), suggesting a significant performance gap at the top.

Notably, one Bahia (BA) seller ranks 5th with 284,903 BRL despite BA 
being only the 7th largest state by customer base — indicating an 
exceptionally high-performing seller worth investigating further.

Surprisingly, Rio de Janeiro (RJ) and Minas Gerais (MG) — the 2nd and 
3rd largest states by customer base — only appear at rankings 13th and 
15th respectively. This underperformance relative to their customer base 
size warrants further investigation into what categories these sellers 
focus on and whether logistics or pricing could be limiting their revenue.

**Recommendation:** Conduct a deeper analysis into the product categories 
sold by top BA, RJ and MG sellers to understand what drives the performance 
gap and identify growth opportunities in these states.

## 10. Average Order Value by Month
**Business Question:** The customer retention team wants to identify which months had the highest average order value, to understand seasonal spending patterns.

In [20]:
df = run("""
WITH order_payments AS (
SELECT order_id,payment_value FROM olist_order_payments_dataset
),
order_month AS (
SELECT order_id,order_purchase_timestamp FROM olist_orders_dataset
)
SELECT AVG(a.payment_value) AS average_payment_value, strftime('%m', b.order_purchase_timestamp) AS month
FROM order_payments AS a JOIN order_month AS b ON a.order_id = b.order_id
GROUP BY month
ORDER BY average_payment_value DESC
""")
df.to_csv('average_payment_value_by_month.csv', index=False)
df

,average_payment_value,month
0,161.511407,09
1,161.408334,04
2,161.228972,10
3,157.676773,05
4,155.774417,06
5,155.523792,03
6,153.263458,07
7,151.962711,11
8,150.855409,08
9,148.994677,01


**Findings:** The top 3 months by average order value — September (161.51 BRL), 
April (161.41 BRL) and October (161.23 BRL) — show nearly identical 
averages, suggesting a seasonal spending pattern possibly driven by 
transitional purchases such as clothing and seasonal appliances. 
Further investigation into product categories sold during these months 
would help confirm this hypothesis.

Surprisingly, December ranks 11th despite being a major gift-giving 
season. While watches_gifts and cool_stuff were top revenue categories 
overall (Problem 5), it appears gift purchases may be concentrated in 
other months — potentially driven by Brazilian cultural events such as 
local festivals or holidays that differ from European seasonal patterns.

**Recommendation:** Conduct a month-by-category breakdown to identify 
exactly which products drive spending in September, April and October. 
Additionally, investigate which months drive gift category purchases to 
better align marketing campaigns with actual Brazilian consumer behavior 
rather than assuming European or global seasonal patterns apply.

## Further Investigation: Top Selling Product Category by Month

In [21]:
df = run("""
SELECT * FROM (
WITH order_items AS (SELECT order_id, product_id FROM olist_order_items_dataset 
),
orders AS (SELECT order_id,order_purchase_timestamp FROM olist_orders_dataset 
),
products AS (SELECT product_id,product_category_name FROM olist_products_dataset 
),
translation AS (SELECT product_category_name, product_category_name_english FROM product_category_name_translation 
)
SELECT  strftime('%m',o.order_purchase_timestamp) AS month , trans.product_category_name_english, COUNT(trans.product_category_name_english) AS total_sold , RANK() OVER (PARTITION BY strftime('%m',o.order_purchase_timestamp) ORDER BY COUNT(trans.product_category_name_english)  DESC) AS ranking
FROM order_items AS oi 
JOIN orders AS o ON oi.order_id = o.order_id
JOIN products AS p ON oi.product_id = p.product_id
JOIN translation AS trans ON p.product_category_name = trans.product_category_name
GROUP BY month, trans.product_category_name_english
) 
WHERE ranking  = 1
""")
df.to_csv('product_category_by_month.csv', index=False)
df

,month,product_category_name_english,total_sold,ranking
0,01,bed_bath_table,896,1
1,02,computers_accessories,1087,1
2,03,bed_bath_table,1087,1
3,04,bed_bath_table,1020,1
4,05,bed_bath_table,1116,1
5,06,bed_bath_table,1155,1
6,07,bed_bath_table,1203,1
7,08,health_beauty,1209,1
8,09,bed_bath_table,533,1
9,10,bed_bath_table,553,1


**Findings:** bed_bath_table dominates as the top selling category in 10 out of 12 
months, further confirming its position as Olist's strongest performing 
category (consistent with Problem 5 revenue findings).

However, two months show notable exceptions:
- **February** — computers_accessories (1,087 units) overtakes bed_bath_table, 
  likely driven by Brazil's Carnival season where tech purchases spike 
  during the extended holiday period
- **August** — health_beauty (1,209 units) leads, coinciding with Dia dos 
  Pais (Brazilian Father's Day), where health and beauty products are 
  commonly purchased as gifts

**Key Takeaway:** Understanding local Brazilian cultural events is critical 
for accurate interpretation of e-commerce trends. Global or European 
seasonal assumptions do not apply to this market, and marketing campaigns 
should be aligned with Brazil's unique cultural calendar.